In [ ]:

# mBERT Amharic Medical Question Classification
# Comparison: Before SMOTE vs After SMOTE
# Outputs: Confusion Matrices, Classification Reports, ROC Curves

import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================


SEED = 42
DATA_PATH = "/content/drive/MyDrive/PHD Folder/QA Model and Dataset/AmHQA2690.csv"
MODEL_NAME = "Davlan/bert-base-multilingual-cased-finetuned-amharic"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5

OUTPUT_DIR = "./mbert_smote_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD AND PREPARE DATA
# ============================================================

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["Question", "Category"]).copy()
df["Question"] = df["Question"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip().str.lower()
df = df[df["Question"] != ""].reset_index(drop=True)

label_names = ["factoid", "list", "definition", "description"]
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label.title() for i, label in enumerate(label_names)}

df = df[df["Category"].isin(label2id)].copy()
df["label"] = df["Category"].map(label2id).astype(int)

print("\nDataset size:", len(df))
print("\nOriginal class distribution:")
print(df["Category"].value_counts().reindex(label_names, fill_value=0))

# Stratified 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# ============================================================
# 3. TOKENIZATION AND DATASET
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QuestionDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["Question"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH
        )
        encoded["labels"] = int(self.labels[idx])
        return encoded

train_dataset = QuestionDataset(train_df)
val_dataset = QuestionDataset(val_df)
test_dataset = QuestionDataset(test_df)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 4. TRAIN mBERT BEFORE SMOTE
# ============================================================

model_before = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/before_smote",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    data_seed=SEED
)

trainer_before = Trainer(
    model=model_before,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

print("\nTraining mBERT BEFORE SMOTE...")
trainer_before.train()

# ============================================================
# 5. PREDICTIONS BEFORE SMOTE
# ============================================================

before_output = trainer_before.predict(test_dataset)
y_test = np.array(test_df["label"])
y_pred_before = np.argmax(before_output.predictions, axis=1)

# Softmax probabilities
def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)

y_prob_before = softmax(before_output.predictions)

# Save the fine-tuned model
trainer_before.save_model(f"{OUTPUT_DIR}/fine_tuned_mbert")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/fine_tuned_mbert")

# ============================================================
# 6. EXTRACT mBERT EMBEDDINGS FOR SMOTE
# ============================================================

# Use the fine-tuned mBERT encoder to extract embeddings.
# SMOTE is applied ONLY to training embeddings.

encoder = trainer_before.model.base_model
encoder.to(device)
encoder.eval()

def extract_embeddings(dataframe, batch_size=32):
    all_embeddings = []

    texts = dataframe["Question"].tolist()

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Extracting embeddings"
    ):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = encoder(**inputs)
            hidden = outputs.last_hidden_state

            # Attention-mask-weighted mean pooling
            mask = inputs["attention_mask"].unsqueeze(-1).float()
            summed = (hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            embeddings = summed / counts

        all_embeddings.append(embeddings.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)

print("\nExtracting training embeddings...")
X_train = extract_embeddings(train_df)

print("\nExtracting validation embeddings...")
X_val = extract_embeddings(val_df)

print("\nExtracting test embeddings...")
X_test = extract_embeddings(test_df)

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

# ============================================================
# 7. APPLY SMOTE TO TRAINING DATA ONLY
# ============================================================

print("\nClass distribution BEFORE SMOTE:")
print(pd.Series(y_train).map(id2label).value_counts())

# k_neighbors is adjusted for small minority classes.
class_counts = pd.Series(y_train).value_counts()
minority_count = int(class_counts.min())

if minority_count < 2:
    raise ValueError(
        "SMOTE requires at least two training examples in every "
        "class. Increase the dataset size or use a different strategy."
    )

smote = SMOTE(
    random_state=SEED,
    k_neighbors=min(5, minority_count - 1)
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nClass distribution AFTER SMOTE:")
print(pd.Series(y_train_smote).map(id2label).value_counts())

# ============================================================
# 8. TRAIN CLASSIFIER AFTER SMOTE
# ============================================================

# Logistic regression learns from the balanced mBERT embeddings.
# Validation and test data remain unchanged.

model_after = LogisticRegression(
    max_iter=2000,
    class_weight=None,
    random_state=SEED
)

print("\nTraining classifier AFTER SMOTE...")
model_after.fit(X_train_smote, y_train_smote)

y_pred_after = model_after.predict(X_test)

# Ensure probability columns match the four original labels
raw_prob_after = model_after.predict_proba(X_test)
y_prob_after = np.zeros((len(X_test), len(label_names)))

for col, class_id in enumerate(model_after.classes_):
    y_prob_after[:, int(class_id)] = raw_prob_after[:, col]

# ============================================================
# 9. CLASSIFICATION REPORTS
# ============================================================

report_before = classification_report(
    y_test,
    y_pred_before,
    labels=list(range(len(label_names))),
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

report_after = classification_report(
    y_test,
    y_pred_after,
    labels=list(range(len(label_names))),
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

print("\n" + "=" * 65)
print("CLASSIFICATION REPORT: BEFORE SMOTE")
print("=" * 65)

print(classification_report(
    y_test,
    y_pred_before,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0
))

print("\n" + "=" * 65)
print("CLASSIFICATION REPORT: AFTER SMOTE")
print("=" * 65)

print(classification_report(
    y_test,
    y_pred_after,
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0
))

# Export class-level precision, recall, F1, and support
report_rows = []

for class_name in label_names:
    for approach, report in [
        ("Before SMOTE", report_before),
        ("After SMOTE", report_after)
    ]:
        report_rows.append({
            "Approach": approach,
            "Class": class_name,
            "Precision": report[class_name]["precision"],
            "Recall": report[class_name]["recall"],
            "F1-score": report[class_name]["f1-score"],
            "Support": report[class_name]["support"]
        })

report_df = pd.DataFrame(report_rows)
report_df.to_csv(
    f"{OUTPUT_DIR}/classification_reports.csv",
    index=False
)

# ============================================================
# 10. CONFUSION MATRICES
# ============================================================

cm_before = confusion_matrix(
    y_test,
    y_pred_before,
    labels=list(range(len(label_names)))
)

cm_after = confusion_matrix(
    y_test,
    y_pred_after,
    labels=list(range(len(label_names)))
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(
    cm_before,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names,
    ax=axes[0]
)

axes[0].set_title("mBERT Before SMOTE")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

sns.heatmap(
    cm_after,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=label_names,
    yticklabels=label_names,
    ax=axes[1]
)

axes[1].set_title("mBERT Embeddings + SMOTE")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/confusion_matrix_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Save confusion matrices
pd.DataFrame(
    cm_before,
    index=label_names,
    columns=label_names
).to_csv(f"{OUTPUT_DIR}/confusion_matrix_before_smote.csv")

pd.DataFrame(
    cm_after,
    index=label_names,
    columns=label_names
).to_csv(f"{OUTPUT_DIR}/confusion_matrix_after_smote.csv")

# ============================================================
# 11. ROC CURVES: MULTICLASS ONE-VS-REST
# ============================================================

y_test_bin = label_binarize(
    y_test,
    classes=list(range(len(label_names)))
)

def plot_multiclass_roc(y_true_bin, y_prob, title, save_path):
    n_classes = y_true_bin.shape[1]

    plt.figure(figsize=(9, 7))

    for i in range(n_classes):
        # ROC is undefined when a test class has only one outcome.
        if len(np.unique(y_true_bin[:, i])) < 2:
            print(f"Skipping ROC for {label_names[i]}: "
                  "test set contains only one outcome.")
            continue

        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )

        class_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{label_names[i]} (AUC = {class_auc:.3f})"
        )

    # Macro-average ROC using interpolated class-specific curves
    valid_classes = [
        i for i in range(n_classes)
        if len(np.unique(y_true_bin[:, i])) >= 2
    ]

    if valid_classes:
        all_fpr = np.unique(np.concatenate([
            roc_curve(y_true_bin[:, i], y_prob[:, i])[0]
            for i in valid_classes
        ]))

        mean_tpr = np.zeros_like(all_fpr)

        for i in valid_classes:
            fpr, tpr, _ = roc_curve(
                y_true_bin[:, i],
                y_prob[:, i]
            )
            mean_tpr += np.interp(all_fpr, fpr, tpr)

        mean_tpr /= len(valid_classes)
        macro_auc = auc(all_fpr, mean_tpr)

        plt.plot(
            all_fpr,
            mean_tpr,
            linestyle="--",
            linewidth=3,
            label=f"Macro-average (AUC = {macro_auc:.3f})"
        )

    plt.plot(
        [0, 1], [0, 1],
        linestyle=":",
        color="black",
        label="Random classifier"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

plot_multiclass_roc(
    y_test_bin,
    y_prob_before,
    "mBERT ROC Curves Before SMOTE",
    f"{OUTPUT_DIR}/roc_before_smote.png"
)

plot_multiclass_roc(
    y_test_bin,
    y_prob_after,
    "mBERT Embeddings + SMOTE ROC Curves",
    f"{OUTPUT_DIR}/roc_after_smote.png"
)

# ============================================================
# 12. PERFORMANCE COMPARISON
# ============================================================

def calculate_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro ROC-AUC": roc_auc_score(
            y_true,
            y_prob,
            labels=list(range(len(label_names))),
            multi_class="ovr",
            average="macro"
        )
    }

results = pd.DataFrame([
    {
        "Approach": "Before SMOTE",
        **calculate_metrics(y_test, y_pred_before, y_prob_before)
    },
    {
        "Approach": "After SMOTE",
        **calculate_metrics(y_test, y_pred_after, y_prob_after)
    }
])

print("\n" + "=" * 65)
print("FINAL PERFORMANCE COMPARISON")
print("=" * 65)
print(results.round(4).to_string(index=False))

results.to_csv(
    f"{OUTPUT_DIR}/performance_comparison.csv",
    index=False
)

# Plot metric comparison
plot_df = results.set_index("Approach")

ax = plot_df.plot(
    kind="bar",
    figsize=(12, 6),
    ylim=(0, 1),
    width=0.75
)

ax.set_title("mBERT Performance Before and After SMOTE")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/performance_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("\nAll outputs saved in:", OUTPUT_DIR)
print("\nGenerated files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(filename)
